# 03 — One-Class SVM

## 1. Intuition

**One-Class SVM tries to learn what “normal data” looks like and then creates a boundary around it.**

Imagine these are normal data points:

```text
        ● ● ●
      ● ● ● ● ●
       ● ● ●
          ●
```

One-Class SVM tries to learn a boundary enclosing the normal region:

```text
       ┌─────────────┐
       │   ● ● ●     │
       │ ● ● ● ● ●   │
       │  ● ● ●      │
       │     ●       │
       └─────────────┘
```

Now a new point appears:

```text
       ┌─────────────┐
       │   ● ● ●     │
       │ ● ● ● ● ●   │
       │  ● ● ●      │
       │     ●       │
       └─────────────┘

                         ×
```

The point `×` is **outside the learned normal region**, so it is flagged as an anomaly.

### Core idea

```text
Normal training data
        ↓
Learn boundary around normal behavior
        ↓
New data point
        ↓
Inside boundary?  → Normal
Outside boundary? → Anomaly
```

### Why is it called **One-Class** SVM?

Because during training, we generally give it **one class: normal data**.

Unlike ordinary classification:

```text
SVM classification:
Normal + Anomaly labels
        ↓
Learn boundary between two classes
```

One-Class SVM:

```text
Mostly/only Normal data
        ↓
Learn boundary around normal behavior
        ↓
Anything sufficiently outside → Anomaly
```

So the fundamental idea is:

> **Isolation Forest isolates unusual points. One-Class SVM learns a boundary around normal points.**


# 2. Learning the Boundary of Normal Data

Now let's understand **what One-Class SVM is actually trying to learn**.

We already know the basic idea:

> One-Class SVM receives normal data and tries to learn a region that represents **normal behavior**.

But the important question is:

> **How does it mathematically create that boundary?**

To understand this, we first need to understand the idea of a **hyperplane**.

---

## Step 1: Imagine normal data in 2D

Suppose every observation has two features:

* \(x_1\) = Feature 1
* \(x_2\) = Feature 2

Our normal data might look like:

```text
x₂
↑
│
│              ●
│         ●  ● ● ●
│       ● ● ● ● ●
│         ● ● ●
│            ●
│
│
└────────────────────→ x₁
```

Most of the normal observations are concentrated in this region.

One-Class SVM wants to separate this **normal region** from the rest of the feature space.

Conceptually:

```text
x₂
↑
│
│              ●
│         ●  ● ● ●
│       ● ● ● ● ●
│         ● ● ●
│            ●
│
│       ┌───────────────┐
│       │  NORMAL       │
│       │   REGION      │
│       └───────────────┘
│
└────────────────────────→ x₁
```

The exact boundary doesn't have to be a rectangle like this. That's just for visualization.

---

# Step 2: SVM thinks in terms of a boundary

A standard SVM tries to find a **hyperplane** that separates classes.

In 2D, a hyperplane is simply a **line**.

For example:

```text
x₂
↑
│
│       ● ● ●
│     ● ● ● ●
│   ● ● ●
│
│───────────────  ← boundary
│
│
└────────────────→ x₁
```

In 3D, the boundary becomes a **plane**.

In higher dimensions, we call it a **hyperplane**.

So:

| Number of features | Boundary   |
| -----------------: | ---------- |
|                  2 | Line       |
|                  3 | Plane      |
|        More than 3 | Hyperplane |

---

# Step 3: But One-Class SVM has a problem

Here's something interesting.

In normal classification, we have two classes:

```text
● ● ● ●       × × × ×
Normal        Anomaly
```

SVM can learn:

> "Put a boundary between these two classes."

But with One-Class SVM, we usually have:

```text
● ● ● ●
● ● ● ●
● ● ●
```

We don't have labeled anomaly points.

So there is nothing on the other side telling the algorithm:

> "This is what an anomaly looks like."

Instead, One-Class SVM asks:

> **Can I find a boundary that encloses the region containing the normal observations?**

That's the fundamental difference.

---

# Step 4: Think of putting the normal data inside a region

Imagine the normal points are objects on a table.

You want to put a boundary around them:

```text
        ● ●
      ● ● ● ●
     ● ● ● ● ●
       ● ● ●

    ┌──────────────┐
    │              │
    │   NORMAL     │
    │    DATA      │
    │              │
    └──────────────┘
```

Then:

```text
Inside → normal
Outside → potentially anomalous
```

This is what One-Class SVM is trying to accomplish.

But it doesn't simply draw the smallest possible box around the points.

It uses an optimization problem to determine an appropriate boundary.

---

# Step 5: The mathematical idea

One-Class SVM uses a decision function of the form:

$$
f(x)=w^T\phi(x)-\rho
$$

Let's introduce each part carefully.

### \(x\)

This is the data point we're evaluating.

For example:

$$
x=
\begin{bmatrix}
5\\
7
\end{bmatrix}
$$

This could mean:

```text
Feature 1 = 5
Feature 2 = 7
```

---

### \(\phi(x)\)

This represents a possible **transformation of the input data into another feature space**.

We'll discuss this properly when we reach **kernels**.

For now, you can think of:

$$
\phi(x)
$$

as:

> "The representation of \(x\) in the space where SVM learns the boundary."

If we're using a simple linear case, this transformation can effectively be the original features.

---

### \(w\)

\(w\) determines the **orientation of the boundary**.

For example:

$$
w=
\begin{bmatrix}
w_1\\
w_2
\end{bmatrix}
$$

It determines how the features contribute to the boundary.

---

### \(\rho\)

\(\rho\) determines the **position of the boundary**.

So, roughly:

```text
w → orientation
ρ → position
```

---

# Step 6: The decision rule

Now we have:

$$
f(x)=w^T\phi(x)-\rho
$$

The sign of this function tells us which side of the boundary the point lies on.

Conceptually:

$$
f(x)>0
$$

→ point is on the **normal side**.

And:

$$
f(x)<0
$$

→ point is on the **anomalous/outside side**.

The boundary itself occurs when:

$$
f(x)=0
$$

So:

```text
                 Boundary
                    ↓
f(x) > 0       f(x)=0       f(x)<0
   │               │             │
   ↓               ↓             ↓
Normal          Boundary       Anomaly
```

This is the mathematical version of:

> **Inside → normal, outside → anomaly.**

---

# Step 7: Tiny numerical example

Suppose, just for understanding, that we have a simple linear decision function:

$$
f(x)=2x_1+x_2-10
$$

Take a point:

$$
x=
\begin{bmatrix}
3\\
5
\end{bmatrix}
$$

Then:

$$
f(x)=2(3)+5-10
$$

$$
=6+5-10
$$

$$
=1
$$

Since:

$$
f(x)>0
$$

the point is classified on the **normal side**.

Now take:

$$
x=
\begin{bmatrix}
1\\
2
\end{bmatrix}
$$

Then:

$$
f(x)=2(1)+2-10
$$

$$
=-6
$$

Since:

$$
f(x)<0
$$

the point is on the **outside/anomalous side**.

**Important:** this is only a simplified numerical example to understand the decision function. Real One-Class SVM training determines \(w\) and \(\rho\) through its optimization procedure, and with nonlinear kernels the boundary can be much more complex.

---

# Step 8: What is One-Class SVM actually optimizing?

This is the deeper idea.

One-Class SVM doesn't simply say:

> "Draw any boundary around all the points."

It tries to find a boundary that captures the **main region of the normal data**, while allowing some observations to fall outside.

Why allow some outside?

Because real datasets aren't perfectly clean.

Suppose:

```text
        ● ● ●
      ● ● ● ● ●
       ● ● ●

                         ●
```

That isolated point might be:

* a genuine anomaly,
* noise,
* measurement error,
* or simply an unusual but valid observation.

So forcing **100% of the training data** inside the boundary isn't necessarily desirable.

This is where the **\(\nu\) (nu) parameter** becomes important.

It controls the trade-off involving how many observations can lie outside the learned region and how complex/tight the boundary can become.

We'll cover \(\nu\) separately.

---

# The complete picture so far

Think about One-Class SVM as this pipeline:

```text
Normal training data
        ↓
Represent data in feature space
        ↓
Find a boundary
        ↓
Boundary separates normal region
        ↓
       New point
        ↓
Evaluate f(x)
        ↓
 ┌───────────────┐
 │               │
f(x) > 0      f(x) < 0
 │               │
 ↓               ↓
Normal        Anomaly
```

And the core mathematical representation is:

$$
\boxed{f(x)=w^T\phi(x)-\rho}
$$

where:

* \(x\) = input data point
* \(\phi(x)\) = representation of \(x\) in feature space
* \(w\) = determines the boundary's orientation
* \(\rho\) = determines the boundary's position


# 2. Learning the Boundary of Normal Data

Now let's understand **what One-Class SVM is actually trying to learn**.

We already know the basic idea:

> One-Class SVM receives normal data and tries to learn a region that represents **normal behavior**.

But the important question is:

> **How does it mathematically create that boundary?**

To understand this, we first need to understand the idea of a **hyperplane**.

---

## Step 1: Imagine normal data in 2D

Suppose every observation has two features:

* \(x_1\) = Feature 1
* \(x_2\) = Feature 2

Our normal data might look like:

```text
x₂
↑
│
│              ●
│         ●  ● ● ●
│       ● ● ● ● ●
│         ● ● ●
│            ●
│
│
└────────────────────→ x₁
```

Most of the normal observations are concentrated in this region.

One-Class SVM wants to separate this **normal region** from the rest of the feature space.

Conceptually:

```text
x₂
↑
│
│              ●
│         ●  ● ● ●
│       ● ● ● ● ●
│         ● ● ●
│            ●
│
│       ┌───────────────┐
│       │  NORMAL       │
│       │   REGION      │
│       └───────────────┘
│
└────────────────────────→ x₁
```

The exact boundary doesn't have to be a rectangle like this. That's just for visualization.

---

# Step 2: SVM thinks in terms of a boundary

A standard SVM tries to find a **hyperplane** that separates classes.

In 2D, a hyperplane is simply a **line**.

For example:

```text
x₂
↑
│
│       ● ● ●
│     ● ● ● ●
│   ● ● ●
│
│───────────────  ← boundary
│
│
└────────────────→ x₁
```

In 3D, the boundary becomes a **plane**.

In higher dimensions, we call it a **hyperplane**.

So:

| Number of features | Boundary   |
| -----------------: | ---------- |
|                  2 | Line       |
|                  3 | Plane      |
|        More than 3 | Hyperplane |

---

# Step 3: But One-Class SVM has a problem

Here's something interesting.

In normal classification, we have two classes:

```text
● ● ● ●       × × × ×
Normal        Anomaly
```

SVM can learn:

> "Put a boundary between these two classes."

But with One-Class SVM, we usually have:

```text
● ● ● ●
● ● ● ●
● ● ●
```

We don't have labeled anomaly points.

So there is nothing on the other side telling the algorithm:

> "This is what an anomaly looks like."

Instead, One-Class SVM asks:

> **Can I find a boundary that encloses the region containing the normal observations?**

That's the fundamental difference.

---

# Step 4: Think of putting the normal data inside a region

Imagine the normal points are objects on a table.

You want to put a boundary around them:

```text
        ● ●
      ● ● ● ●
     ● ● ● ● ●
       ● ● ●

    ┌──────────────┐
    │              │
    │   NORMAL     │
    │    DATA      │
    │              │
    └──────────────┘
```

Then:

```text
Inside → normal
Outside → potentially anomalous
```

This is what One-Class SVM is trying to accomplish.

But it doesn't simply draw the smallest possible box around the points.

It uses an optimization problem to determine an appropriate boundary.

---

# Step 5: The mathematical idea

One-Class SVM uses a decision function of the form:

$$
f(x)=w^T\phi(x)-\rho
$$

Let's introduce each part carefully.

### \(x\)

This is the data point we're evaluating.

For example:

$$
x=
\begin{bmatrix}
5\\
7
\end{bmatrix}
$$

This could mean:

```text
Feature 1 = 5
Feature 2 = 7
```

---

### \(\phi(x)\)

This represents a possible **transformation of the input data into another feature space**.

We'll discuss this properly when we reach **kernels**.

For now, you can think of:

$$
\phi(x)
$$

as:

> "The representation of \(x\) in the space where SVM learns the boundary."

If we're using a simple linear case, this transformation can effectively be the original features.

---

### \(w\)

\(w\) determines the **orientation of the boundary**.

For example:

$$
w=
\begin{bmatrix}
w_1\\
w_2
\end{bmatrix}
$$

It determines how the features contribute to the boundary.

---

### \(\rho\)

\(\rho\) determines the **position of the boundary**.

So, roughly:

```text
w → orientation
ρ → position
```

---

# Step 6: The decision rule

Now we have:

$$
f(x)=w^T\phi(x)-\rho
$$

The sign of this function tells us which side of the boundary the point lies on.

Conceptually:

$$
f(x)>0
$$

→ point is on the **normal side**.

And:

$$
f(x)<0
$$

→ point is on the **anomalous/outside side**.

The boundary itself occurs when:

$$
f(x)=0
$$

So:

```text
                 Boundary
                    ↓
f(x) > 0       f(x)=0       f(x)<0
   │               │             │
   ↓               ↓             ↓
Normal          Boundary       Anomaly
```

This is the mathematical version of:

> **Inside → normal, outside → anomaly.**

---

# Step 7: Tiny numerical example

Suppose, just for understanding, that we have a simple linear decision function:

$$
f(x)=2x_1+x_2-10
$$

Take a point:

$$
x=
\begin{bmatrix}
3\\
5
\end{bmatrix}
$$

Then:

$$
f(x)=2(3)+5-10
$$

$$
=6+5-10
$$

$$
=1
$$

Since:

$$
f(x)>0
$$

the point is classified on the **normal side**.

Now take:

$$
x=
\begin{bmatrix}
1\\
2
\end{bmatrix}
$$

Then:

$$
f(x)=2(1)+2-10
$$

$$
=-6
$$

Since:

$$
f(x)<0
$$

the point is on the **outside/anomalous side**.

**Important:** this is only a simplified numerical example to understand the decision function. Real One-Class SVM training determines \(w\) and \(\rho\) through its optimization procedure, and with nonlinear kernels the boundary can be much more complex.

---

# Step 8: What is One-Class SVM actually optimizing?

This is the deeper idea.

One-Class SVM doesn't simply say:

> "Draw any boundary around all the points."

It tries to find a boundary that captures the **main region of the normal data**, while allowing some observations to fall outside.

Why allow some outside?

Because real datasets aren't perfectly clean.

Suppose:

```text
        ● ● ●
      ● ● ● ● ●
       ● ● ●

                         ●
```

That isolated point might be:

* a genuine anomaly,
* noise,
* measurement error,
* or simply an unusual but valid observation.

So forcing **100% of the training data** inside the boundary isn't necessarily desirable.

This is where the **\(\nu\) (nu) parameter** becomes important.

It controls the trade-off involving how many observations can lie outside the learned region and how complex/tight the boundary can become.

We'll cover \(\nu\) separately.

---

# The complete picture so far

Think about One-Class SVM as this pipeline:

```text
Normal training data
        ↓
Represent data in feature space
        ↓
Find a boundary
        ↓
Boundary separates normal region
        ↓
       New point
        ↓
Evaluate f(x)
        ↓
 ┌───────────────┐
 │               │
f(x) > 0      f(x) < 0
 │               │
 ↓               ↓
Normal        Anomaly
```

And the core mathematical representation is:

$$
\boxed{f(x)=w^T\phi(x)-\rho}
$$

where:

* \(x\) = input data point
* \(\phi(x)\) = representation of \(x\) in feature space
* \(w\) = determines the boundary's orientation
* \(\rho\) = determines the boundary's position

### The key idea to remember

> **One-Class SVM learns a decision boundary from normal data. A new point is judged by which side of that learned boundary it falls on.**


# 3. Hyperplane & Decision Boundary

Now let's understand **what the boundary actually is** in One-Class SVM.

We previously saw:

$$
f(x)=w^T\phi(x)-\rho
$$

The boundary is where:

$$
f(x)=0
$$

So:

$$
w^T\phi(x)-\rho=0
$$

or:

$$
\boxed{w^T\phi(x)=\rho}
$$

This equation represents the **decision boundary**.

---

## 1. What is a hyperplane?

The word *hyperplane* sounds complicated, but the idea is simple.

| Number of dimensions | Boundary   |
| -------------------- | ---------- |
| 1D                   | Point      |
| 2D                   | Line       |
| 3D                   | Plane      |
| Higher dimensions    | Hyperplane |

For example, in 2D:

$$
w_1x_1+w_2x_2=\rho
$$

is a line.

Imagine:

```text
x₂
↑
│       ● ● ●
│     ● ● ● ●
│    ● ● ●
│
│────────────────  ← decision boundary
│
│
└────────────────→ x₁
```

The boundary divides the space into two regions.

---

## 2. What does \(w\) do?

Remember:

$$
w=
\begin{bmatrix}
w_1\\
w_2
\end{bmatrix}
$$

The vector \(w\) determines the **orientation of the boundary**.

For example, consider:

$$
x_1+x_2=10
$$

Here:

$$
w=
\begin{bmatrix}
1\\
1
\end{bmatrix}
$$

This produces a diagonal boundary.

```text
x₂
↑
│        ╲
│         ╲
│          ╲
│           ╲
│            ╲
│             ╲
└────────────────→ x₁
```

If the values inside \(w\) change, the orientation can change.

So you can remember:

> **\(w\) controls the direction/orientation of the boundary.**

---

## 3. What does \(\rho\) do?

Now consider:

$$
x_1+x_2=10
$$

If we change it to:

$$
x_1+x_2=15
$$

the boundary has the **same orientation**, but it moves to another position.

That's the role of \(\rho\).

```text
       boundary 2
          ╲
           ╲

    boundary 1
       ╲
        ╲
```

So:

> **\(w\) → orientation**
> **\(\rho\) → position**

---

## 4. How does a point get classified?

Once the boundary is learned, we calculate:

$$
f(x)=w^T\phi(x)-\rho
$$

Then look at its sign.

### If:

$$
f(x)>0
$$

the point is on one side of the boundary → **normal**.

### If:

$$
f(x)<0
$$

the point is on the other side → **anomaly**.

### If:

$$
f(x)=0
$$

the point lies exactly on the **decision boundary**.

So:

```text
             Decision Boundary
                    │
                    ↓
       Normal       │       Anomaly
         ● ● ●      │          ×
       ● ● ● ●      │
         ● ●        │
                    │
                  f(x)=0
```

---

## 5. Tiny numerical example

Suppose:

$$
f(x)=x_1+x_2-10
$$

The decision boundary is:

$$
x_1+x_2-10=0
$$

Therefore:

$$
x_1+x_2=10
$$

Take:

$$
x=
\begin{bmatrix}
4\\
5
\end{bmatrix}
$$

Then:

$$
f(x)=4+5-10=-1
$$

So this point is on the negative side.

Now take:

$$
x=
\begin{bmatrix}
6\\
7
\end{bmatrix}
$$

$$
f(x)=6+7-10=3
$$

This point is on the positive side.

The exact interpretation of positive/negative depends on the One-Class SVM convention, but conceptually the **sign tells us which side of the learned boundary the point lies on**.

---

## 6. Why do we need kernels?

Here's the limitation of what we've just seen.

A simple hyperplane gives us a **linear boundary**.

But real normal data may look like this:

```text
        ● ● ●
      ●       ●
     ●    ×    ●
      ●       ●
        ● ● ●
```

The normal data surrounds the center.

A straight line cannot properly enclose this structure.

That's where **kernels** become important.

Instead of forcing One-Class SVM to learn a simple linear boundary in the original feature space, a kernel allows it to effectively work in a transformed feature space where a more complex boundary can be learned.

That is the reason the next concepts—**Support Vectors and Kernel Idea**—are important.

### Main takeaway

$$
\boxed{w^T\phi(x)=\rho}
$$

is the learned decision boundary.

* \(w\) → orientation
* \(\rho\) → position
* \(f(x)\) → tells which side the point lies on
* Kernel → allows the boundary to become nonlinear/complex.


# 4. Support Vectors

Now we need to understand **which points actually determine the boundary** learned by One-Class SVM.

The key idea is:

> **Not every training point is equally important for defining the boundary. A smaller set of points near the boundary has the strongest influence. These are called support vectors.**

---

## 1. Imagine the normal data

Suppose our normal data looks like this:

```text
x₂
↑
│
│          ● ●
│       ● ● ● ● ●
│      ● ● ● ● ●
│       ● ● ● ●
│          ●
│
└────────────────────→ x₁
```

One-Class SVM wants to learn a boundary around this normal region.

Conceptually:

```text
x₂
↑
│       ┌─────────────┐
│       │   ● ●       │
│       │ ● ● ● ● ●   │
│       │● ● ● ● ●    │
│       │ ● ● ● ●     │
│       │    ●        │
│       └─────────────┘
└────────────────────────→ x₁
```

Some points are **deep inside** the normal region.

Others are **close to the boundary**.

The points close to the boundary are especially important.

---

# 2. What are support vectors?

Consider:

```text
             Boundary
                 ↓
        ┌─────────────────┐
        │                 │
      ● │ ● ● ● ● ● ●   │
        │                 │
        │                 │
        └─────────────────┘
```

The points close to the boundary help determine **where that boundary should be placed**.

These important points are called:

$$
\boxed{\text{Support Vectors}}
$$

You can think of them as:

> **The points that "support" or help define the learned boundary.**

---

# 3. Why aren't points in the middle as important?

Suppose we have:

```text
          ● ●
       ●       ●
      ●         ●
      ●         ●
       ●       ●
          ● ●
```

A point deep in the center:

```text
          ● ●
       ●       ●
      ●    ★    ●
      ●         ●
       ●       ●
          ● ●
```

is obviously normal.

Moving that point slightly usually doesn't require the boundary to change much.

But consider a point near the edge:

```text
          ● ●
       ●       ●
      ●         ●
      ●         ● ● ← boundary-sensitive point
       ●       ●
          ● ●
```

Moving that point can affect where the boundary should be.

Therefore:

> **Points near the boundary have more influence on the boundary than points deep inside the normal region.**

---

# 4. Connection with ordinary SVM

This idea comes from ordinary SVM as well.

In standard classification:

```text
● ● ● ●       │       × × × ×
● ● ●         │       × × ×
● ● ● ●       │       × × ×
              ↑
          boundary
```

The points closest to the separating boundary are the **support vectors**.

They determine the position of the separating hyperplane.

One-Class SVM uses the same fundamental idea, but instead of separating:

```text
Normal vs Anomaly
```

it learns a boundary around the **normal data**.

---

# 5. Mathematical connection

Recall our decision function:

$$
f(x)=w^T\phi(x)-\rho
$$

The boundary occurs when:

$$
f(x)=0
$$

Therefore:

$$
w^T\phi(x)=\rho
$$

Points close to this boundary have decision-function values close to zero.

So conceptually:

$$
f(x)\approx0
$$

means:

> The point is close to the learned boundary.

These boundary-related points are important in determining the solution.

---

# 6. Why support vectors matter for anomaly detection

Imagine this:

```text
             Normal region
          ┌───────────────┐
          │ ● ● ● ● ●     │
          │ ● ● ● ● ●     │
          │ ● ● ● ●       │
          └───────────────┘
                ↑
          support vectors
```

The support vectors essentially help define:

> **"This is approximately where normal behavior ends."**

Then when a new point arrives:

```text
          ┌───────────────┐
          │ ● ● ● ● ●     │
          │ ● ● ● ● ●     │
          │ ● ● ● ●       │
          └───────────────┘

                         ×
```

the model evaluates where that point lies relative to the learned boundary.

If it's sufficiently outside:

$$
f(x)<0
$$

→ anomaly.

---

# 7. Important distinction

Don't think:

> "Support vectors are the anomalies."

That's **not correct**.

A support vector is simply a training point that plays an important role in defining the boundary.

A support vector can be:

* a normal point near the edge,
* a point that the model allows outside the main region,
* or another boundary-related observation depending on the learned solution.

So:

$$
\boxed{\text{Support Vector} \neq \text{Anomaly}}
$$

Instead:

$$
\boxed{\text{Support Vectors} \rightarrow \text{Help define the boundary}}
$$

and then:

$$
\boxed{\text{Boundary} \rightarrow \text{Classifies new observations}}
$$

---

## The mental model

Keep this chain in mind:

```text
Normal training data
        ↓
Some points are near the edge
        ↓
Those points strongly influence the boundary
        ↓
Those are support vectors
        ↓
Boundary is learned
        ↓
New point is evaluated against boundary
        ↓
Normal / Anomaly
```

### Main takeaway

> **Support vectors are the important training observations that help determine the One-Class SVM boundary, especially those close to the boundary. They are not automatically anomalies.**


# 5. Kernel Idea

Now we come to one of the **most important parts of One-Class SVM**.

The problem is that a simple hyperplane can only create a **linear boundary**.

But normal data can have a complicated shape.

---

## 1. Why a straight boundary may not work

Suppose our normal data looks like this:

```text
x₂
↑
│
│        ● ● ●
│      ●       ●
│     ●         ●
│     ●         ●
│      ●       ●
│        ● ● ●
│
└──────────────────→ x₁
```

The normal observations form something like a **circle/ring**.

A straight line cannot properly surround this region:

```text
       ● ● ●
     ●       ●
    ●         ●
    ●         ●
     ●       ●
       ● ● ●

───────────────  ← straight line
```

We need a **nonlinear boundary**.

---

# 2. The basic idea of a kernel

Instead of trying to learn a complicated boundary directly in the original feature space, we can **transform the data into another feature space**.

Remember our previous equation:

$$
f(x)=w^T\phi(x)-\rho
$$

Here:

$$
\phi(x)
$$

represents the transformed version of the data.

The idea is:

```text
Original feature space
        ↓
   transformation
        ↓
New feature space
        ↓
Linear boundary
```

Something that looks complicated in the original space may become easier to separate in the transformed space.

---

# 3. Simple example

Suppose our data has two features:

$$
x=
\begin{bmatrix}
x_1\\
x_2
\end{bmatrix}
$$

Imagine the normal data forms a circle.

Instead of using only \(x_1\) and \(x_2\), we could introduce another feature:

$$
x_3=x_1^2+x_2^2
$$

Now we're representing the data using:

$$
\phi(x)=
\begin{bmatrix}
x_1\\
x_2\\
x_1^2+x_2^2
\end{bmatrix}
$$

The circular structure can become much easier to represent in this new space.

The important insight is:

> **A nonlinear structure in the original space can become a simpler, potentially linear structure in a transformed feature space.**

---

# 4. But explicitly transforming the data can be expensive

Here's where the **kernel trick** comes in.

Instead of explicitly calculating:

$$
\phi(x)
$$

for every point, kernels allow us to calculate the similarity between transformed points directly.

A kernel function is written as:

$$
K(x_i,x_j)
$$

It measures a certain notion of similarity between two observations in the transformed feature space.

So instead of explicitly doing:

$$
\phi(x_i)^T\phi(x_j)
$$

we can use:

$$
\boxed{K(x_i,x_j)}
$$

This is the basic idea behind the **kernel trick**.

---

# 5. RBF kernel

One-Class SVM commonly uses the **RBF (Radial Basis Function) kernel**.

Its mathematical form is:

$$
K(x_i,x_j)
=
\exp\left(-\gamma\|x_i-x_j\|^2\right)
$$

Let's understand the variables.

* \(x_i\) = first data point
* \(x_j\) = second data point
* \(\|x_i-x_j\|^2\) = squared distance between them
* \(\gamma\) = controls how quickly similarity decreases with distance

The important intuition is:

> **Nearby points have high similarity; far-away points have low similarity.**

For example:

```text
A ●────● B
  close

A ●────────────────────● C
          far
```

Then roughly:

$$
K(A,B) > K(A,C)
$$

because A and B are closer.

---

# 6. What does `gamma` do?

This is important because we'll later see `gamma` as a One-Class SVM hyperparameter.

### Small \(\gamma\)

Similarity decreases **slowly** with distance.

So each point can influence a relatively larger region.

Conceptually:

```text
      large influence
          ↓
       ┌─────┐
     ┌─┘  ●  └─┐
     │         │
     └─────────┘
```

This tends to produce a **smoother, less sensitive boundary**.

---

### Large \(\gamma\)

Similarity decreases **very quickly** with distance.

Each point has a more localized influence.

Conceptually:

```text
      small influence
           ↓
          ┌─┐
          │●│
          └─┘
```

This can produce a **more flexible/complex boundary** and may make the model sensitive to individual points.

So:

$$
\boxed{\text{Small }\gamma \rightarrow \text{smoother boundary}}
$$

$$
\boxed{\text{Large }\gamma \rightarrow \text{more complex boundary}}
$$

---

# 7. Linear vs RBF

The difference can be visualized like this:

### Linear kernel

```text
● ● ● ●
● ● ● ●
● ● ● ●

──────────────
```

Simple boundary.

### RBF kernel

```text
       ● ● ●
     ●       ●
    ●         ●
     ●       ●
       ● ● ●

     ╭───────╮
     │       │
     ╰───────╯
```

Can represent much more complex boundaries.

---

# 8. Why does this matter for anomaly detection?

Imagine normal observations form a complicated region:

```text
          ● ●
       ●       ●
      ●         ●
       ● ● ● ● ●
          ●
```

A linear One-Class SVM may struggle to describe this shape.

With an RBF kernel, the model can learn a **nonlinear boundary** that follows the structure of the normal data much more effectively.

So the overall process becomes:

```text
Normal data
     ↓
Kernel representation
     ↓
Learn boundary
     ↓
Complex normal region
     ↓
New point
     ↓
Inside / outside
     ↓
Normal / anomaly
```

---

## One important clarification

The kernel does **not** mean:

> "The model simply draws circles around every point."

That's an oversimplification.

The kernel changes how observations are represented/similarities are computed, allowing the SVM optimization to produce a nonlinear decision boundary.

---

### Main takeaway

> **Kernel = a way to let One-Class SVM learn nonlinear boundaries.**

And the most important one for us is:

$$
\boxed{
K(x_i,x_j)
=
\exp\left(-\gamma\|x_i-x_j\|^2\right)
}
$$

with:

* **RBF kernel** → flexible nonlinear boundary
* **small \(\gamma\)** → smoother boundary
* **large \(\gamma\)** → more complex/local boundary


# 6. ν (Nu) Parameter

Now we come to one of the **most important hyperparameters in One-Class SVM**.

The parameter is written as:

$$
\boxed{\nu}
$$

and is pronounced **"nu"**.

It mainly controls **how much of the training data the model is willing to consider as outside the normal region**, while also affecting how tight/complex the boundary becomes.

---

## 1. Intuition

Imagine we have 100 training observations:

```text
95 points → tightly grouped
5 points  → somewhat far away
```

One-Class SVM has to decide:

> "Should I make the boundary large enough to include all 100 points?"

That might not be a good idea, because those 5 points could be noise or unusual observations.

Instead, we might allow some points to fall outside the learned normal region.

That's what \(\nu\) helps control.

---

## 2. Example

Suppose:

```python
model = OneClassSVM(nu=0.05)
```

Here:

$$
\nu=0.05=5\%
$$

So the model allows for roughly **5% of the training observations to be treated as outliers**.

With 100 observations:

$$
100\times0.05=5
$$

So approximately 5 observations may fall outside the learned normal region.

Conceptually:

```text
        Normal region
     ┌─────────────────┐
     │ ● ● ● ● ● ●     │
     │ ● ● ● ● ● ●     │
     │ ● ● ● ● ●       │
     │ ● ● ● ● ●       │
     └─────────────────┘

       ×       ×
                ×       ← points allowed outside
```

---

## 3. What happens if we increase \(\nu\)?

Suppose:

$$
\nu=0.01
$$

versus:

$$
\nu=0.20
$$

### Small \(\nu\)

```text
ν = 0.01
```

The model expects very few outliers.

So the boundary tends to be **more inclusive** of the training data.

### Larger \(\nu\)

```text
ν = 0.20
```

The model allows more observations to be treated as outside.

So the boundary can become **tighter around the main normal region**.

Conceptually:

```text
Small ν:

      ┌───────────────┐
      │ ● ● ● ● ●     │
      │ ● ● ● ● ●     │
      │ ● ● ● ●       │
      └───────────────┘


Large ν:

        ┌───────────┐
        │ ● ● ● ●   │
        │ ● ● ● ●   │
        │ ● ● ●     │
        └───────────┘
```

The exact shape also depends strongly on the kernel and `gamma`.

---

## 4. Important relationship with contamination

Don't confuse **`nu` in One-Class SVM** with **`contamination` in Isolation Forest**.

They are not exactly the same parameter, but they have a related purpose.

| Isolation Forest               | One-Class SVM                                                   |
| ------------------------------ | --------------------------------------------------------------- |
| `contamination`                | `nu`                                                            |
| Helps determine anomaly cutoff | Controls allowed outliers/boundary behavior                     |
| Expected anomaly proportion    | Upper bound on training errors / lower bound on support vectors |

For practical learning:

> **Both are related to how aggressively the model treats observations as anomalous, but their mathematical roles are different.**

---

## 5. Another important role of \(\nu\)

\(\nu\) also has a theoretical relationship with the **fraction of support vectors**.

In One-Class SVM:

$$
\boxed{\nu \text{ is a lower bound on the fraction of support vectors}}
$$

and:

$$
\boxed{\nu \text{ is an upper bound on the fraction of training errors}}
$$

You don't need to memorize the proof right now. The practical intuition is:

> Increasing \(\nu\) allows the model to treat more observations as outside the normal region and generally results in more support-vector involvement.

---

## 6. Choosing `nu`

There isn't one universally correct value.

For example:

```python
OneClassSVM(nu=0.01)
```

might make sense if you expect very few anomalies.

Whereas:

```python
OneClassSVM(nu=0.10)
```

allows a substantially larger fraction of observations to be considered unusual.

So choose it based on:

* expected anomaly rate
* domain knowledge
* validation/testing
* desired sensitivity

---

### Main takeaway

$$
\boxed{\nu \rightarrow \text{controls how much abnormality/outside data the model allows}}
$$

Remember the practical distinction:

> **Isolation Forest `contamination` → expected proportion of anomalies used for the final decision threshold.**

> **One-Class SVM `nu` → controls the allowed training errors/outliers and the support-vector/boundary behavior.**



# 7. Anomaly / Decision Score

Now we need to understand **how One-Class SVM actually decides whether a new point is normal or anomalous**.

We already have the decision function:

$$
f(x)=w^T\phi(x)-\rho
$$

This function gives us a **score** for the point.

---

## 1. What does the score tell us?

The score tells us **where the point lies relative to the learned boundary**.

Remember:

$$
f(x)=0
$$

is the decision boundary.

So conceptually:

```text id="1jv7tq"
                 Boundary
                    │
                    ↓
       Normal       │       Anomaly
                    │
     ●              │              ×
  ●     ●           │
     ●              │
                    │
                  f(x)=0
```

A point can therefore have:

* **positive score** → normal side
* **score near 0** → close to boundary
* **negative score** → anomaly side

---

## 2. Why is the distance from zero useful?

Suppose we have three points:

$$
f(A)=5
$$

$$
f(B)=0.2
$$

$$
f(C)=-3
$$

Then:

```text id="d3r1b5"
A → 5       far on normal side
B → 0.2     close to boundary
C → -3      anomaly side
```

So `B` is interesting because it's close to the boundary.

Conceptually:

$$
|f(x)| \text{ small}
$$

means the point is close to the decision boundary.

---

# 3. Visualizing the idea

Imagine the learned boundary:

```text id="xq5s5p"
Normal region                    Outside

   ● ● ● ●
 ● ● ● ● ● ●
   ● ● ● ●       │       ×
                  │
                  │
             boundary
```

Now imagine moving a point from inside toward the boundary:

```text id="w8u0m0"
       ●
       ↓
       ↓
       ↓
       │
       │ boundary
       │
```

Its decision score moves toward zero.

So:

```text id="l9tj85"
Far inside normal region
        ↓
     positive score
        ↓
Closer to boundary
        ↓
   score ≈ 0
        ↓
Outside boundary
        ↓
   negative score
```

---

# 4. `decision_function()` in sklearn

In scikit-learn, we can obtain this score using:

```python
scores = model.decision_function(X)
```

For example:

```python
from sklearn.svm import OneClassSVM

model = OneClassSVM(
    kernel="rbf",
    nu=0.05,
    gamma="scale"
)

model.fit(X)

scores = model.decision_function(X)
predictions = model.predict(X)
```

The important part is:

```python
model.decision_function(X)
```

This gives the **decision score relative to the learned boundary**.

---

# 5. `predict()` then converts this into a decision

After calculating the decision score, the model produces a prediction.

In scikit-learn:

$$
\boxed{+1=\text{Normal}}
$$

$$
\boxed{-1=\text{Anomaly}}
$$

So conceptually:

```text id="n5t0k1"
             decision_function
                    ↓
                 score
                    ↓
             ┌──────┴──────┐
             ↓             ↓
        positive        negative
             ↓             ↓
          Normal        Anomaly
             +1            -1
```

---

# 6. Tiny example

Suppose the model produces:

| Point | Decision score | Prediction |
| ----- | -------------: | ---------: |
| A     |            2.5 |         +1 |
| B     |            1.2 |         +1 |
| C     |            0.1 |         +1 |
| D     |           -0.8 |         -1 |
| E     |           -2.4 |         -1 |

So:

```text
A → normal
B → normal
C → normal but close to boundary
D → anomaly
E → anomaly
```

Point `C` is worth paying attention to because its score is close to zero.

It is not necessarily an anomaly, but it is **close to the learned normal boundary**.

---

# 7. Important distinction: score vs anomaly probability

This is important.

The One-Class SVM decision score is **not a probability**.

For example:

$$
f(x)=0.8
$$

does **not** mean:

> "There is an 80% probability that this point is normal."

It is a decision score indicating the point's position relative to the learned decision boundary.

So don't interpret it like:

```text
0.8 = 80% normal
0.2 = 20% normal
```

That's incorrect.

---

# 8. Connecting everything we've learned

We can now connect the entire One-Class SVM process:

```text id="1ny4jq"
Normal training data
        ↓
Learn boundary
        ↓
Support vectors help define boundary
        ↓
Kernel allows nonlinear boundary
        ↓
ν controls boundary/outlier behavior
        ↓
New data point
        ↓
Calculate decision score
        ↓
       f(x)
        ↓
 ┌──────────────┐
 │              │
positive      negative
 │              │
 ↓              ↓
Normal       Anomaly
```

The core equation remains:

$$
\boxed{f(x)=w^T\phi(x)-\rho}
$$

And the key idea is:

> **The decision score tells us which side of the learned normal boundary a point lies on. `predict()` converts that into `+1` (normal) or `-1` (anomaly).**


# 8. Numerical Example — One-Class SVM

Now let's put the concepts together with a **small numerical example**.

We'll keep it simple enough to calculate by hand, while still showing the complete flow.

---

## Step 1: Normal training data

Suppose we have one feature \(x\), and these are our normal observations:

$$
X=
\begin{bmatrix}
2\\
3\\
4\\
5\\
6
\end{bmatrix}
$$

So visually:

```text
2   3   4   5   6
●   ●   ●   ●   ●
```

These are the observations we give to One-Class SVM as **normal training data**.

Now suppose two new observations arrive:

$$
x_A=4
$$

and

$$
x_B=10
$$

Intuitively:

```text
Training data:

2   3   4   5   6
●   ●   ●   ●   ●


New points:

4  → close to normal data
10 → far from normal data
```

We expect:

$$
x_A=4 \rightarrow \text{Normal}
$$

$$
x_B=10 \rightarrow \text{Anomaly}
$$

---

# Step 2: The model learns a normal boundary

One-Class SVM looks at the training data:

```text
2   3   4   5   6
●   ●   ●   ●   ●
```

and learns a region representing normal behavior.

For illustration, imagine it learns approximately:

```text
       Normal region
      ┌─────────────┐
      │             │
      └─────────────┘
      2             6
```

The exact boundary is determined by the SVM optimization, `nu`, kernel, `gamma`, etc.

---

# Step 3: Evaluate the new point \(x_A=4\)

The model calculates its decision function:

$$
f(x)=w^T\phi(x)-\rho
$$

Suppose, **for illustration**, the trained model gives:

$$
f(4)=0.8
$$

Since:

$$
f(4)>0
$$

the point is classified as:

$$
\boxed{\text{Normal}}
$$

---

# Step 4: Evaluate \(x_B=10\)

Now:

$$
x_B=10
$$

Suppose the model gives:

$$
f(10)=-1.7
$$

Since:

$$
f(10)<0
$$

the point is classified as:

$$
\boxed{\text{Anomaly}}
$$

So:

| Point  | Decision score | Result  |
| ------ | -------------: | ------- |
| \(4\)  |       \(+0.8\) | Normal  |
| \(10\) |       \(-1.7\) | Anomaly |

---

# Step 5: What happened internally?

The complete process was:

```text
Normal training data
2, 3, 4, 5, 6
       ↓
Learn normal region
       ↓
Learn boundary
       ↓
New point
       ↓
Calculate decision score
       ↓
 ┌──────────────┐
 │              │
f(x) > 0     f(x) < 0
 │              │
 ↓              ↓
Normal       Anomaly
```

---

## One important correction about this numerical example

The values:

$$
f(4)=0.8
$$

and

$$
f(10)=-1.7
$$

were **illustrative**, not manually derived from the training data.

That's because actually training a One-Class SVM requires solving its optimization problem to obtain the learned parameters and, with kernels, the decision function depends on the support vectors and kernel calculations.

The important thing we're learning here is the **decision process**:

$$
\boxed{\text{Training data}
\rightarrow
\text{Boundary}
\rightarrow
\text{Decision score}
\rightarrow
\text{Normal/Anomaly}}
$$

The actual Python implementation will let us train the model and obtain the real scores.


# 9. Python Implementation — One-Class SVM

Now let's implement what we learned using **scikit-learn**.

We'll use a small dataset where most points are normal and a few are clearly different.

### Step 1: Create the data

```python
import numpy as np

X = np.array([
    [10, 20],
    [11, 21],
    [12, 19],
    [13, 22],
    [14, 20],
    [15, 23],
    [11, 20],
    [13, 21],
    [14, 22],
    [30, 50]
])
```

Here:

* First 9 observations are relatively close together.
* `[30, 50]` is far away and is a potential anomaly.

---

## Step 2: Create the model

```python
from sklearn.svm import OneClassSVM

model = OneClassSVM(
    kernel="rbf",
    nu=0.1,
    gamma="scale"
)
```

### What we specified

```text
kernel="rbf"
    ↓
Allows nonlinear boundary

nu=0.1
    ↓
Controls outlier/boundary behavior

gamma="scale"
    ↓
Controls how localized the RBF influence is
```

---

## Step 3: Train the model

```python
model.fit(X)
```

The model now uses the training data to learn:

* the decision boundary
* support vectors
* parameters needed for classification

---

## Step 4: Get predictions

```python
predictions = model.predict(X)
```

One-Class SVM uses:

$$
+1=\text{Normal}
$$

$$
-1=\text{Anomaly}
$$

You can inspect them:

```python
print(predictions)
```

You might get something conceptually like:

```text
[ 1  1  1  1  1  1  1  1  1 -1]
```

The exact result can depend on the chosen hyperparameters.

So:

```text
[10,20] → Normal
[11,21] → Normal
...
[14,22] → Normal
[30,50] → Anomaly
```

---

# Step 5: Get the decision scores

Instead of only asking:

> "Normal or anomaly?"

we can look at **how the model evaluates each point**.

```python
scores = model.decision_function(X)

print(scores)
```

Conceptually:

```text
Point       Score
--------------------
[10,20]     +0.42
[11,21]     +0.51
[12,19]     +0.37
...
[30,50]     -0.83
```

Remember:

$$
f(x)>0 \rightarrow \text{normal}
$$

$$
f(x)<0 \rightarrow \text{anomaly}
$$

A score close to zero means the point is close to the learned boundary.

---

# Step 6: Combine everything

A useful way to inspect the results is:

```python
for point, score, prediction in zip(X, scores, predictions):
    print(point, score, prediction)
```

You can think of the output as:

```text
Data Point       Score       Prediction
-----------------------------------------
[10 20]          positive       +1
[11 21]          positive       +1
[12 19]          positive       +1
...
[30 50]          negative       -1
```

So the complete workflow is:

```text
X
↓
OneClassSVM
↓
fit()
↓
Learn normal boundary
↓
decision_function()
↓
Decision score
↓
predict()
↓
+1 Normal / -1 Anomaly
```

### One important practical point

For a real dataset, **feature scaling is usually important for One-Class SVM**, especially with the RBF kernel, because the kernel uses distances between observations.

For example:

```python
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

model.fit(X_scaled)
```

This prevents a feature with a much larger numerical scale from dominating the distance calculations.

So for the practical implementation, remember:

$$
\boxed{\text{Scale features} \rightarrow \text{Train One-Class SVM} \rightarrow \text{Score} \rightarrow \text{Predict}}
$$


## 10. Important Hyperparameters

### `kernel`

Controls the **type of boundary** One-Class SVM learns.

Common choices:

```python
kernel="linear"
```

→ linear boundary.

```python
kernel="rbf"
```

→ nonlinear, flexible boundary.

For anomaly detection, **`rbf` is commonly useful** when the normal data has a complex shape.

**Remember:** `kernel` → **What type of boundary can the model learn?**

### `nu`

Controls the **allowed proportion of training outliers** and influences the boundary.

```python
nu=0.05
```

→ roughly **5%** of training observations can be treated as outliers.

* Smaller `nu` → fewer outliers allowed, generally a more inclusive boundary.
* Larger `nu` → more outliers allowed, generally a tighter boundary.

**Remember:** `nu` → **How much training data can be considered outside the normal region?**


### `gamma`

Controls **how strongly each training point influences the decision boundary** when using the RBF kernel.

```python
gamma=0.1
```

* **Small `gamma`** → wider influence → smoother, simpler boundary.
* **Large `gamma`** → local influence → more complex, sensitive boundary.

```text
Small gamma → smooth boundary
Large gamma → complex boundary
```

**Remember:** `gamma` → **How local/flexible is the boundary?**


That completes the **Important Hyperparameters**. ✅

### One-Class SVM recap

The complete flow is:

```text
Normal Data
    ↓
Learn normal boundary
    ↓
Support Vectors help define boundary
    ↓
Kernel allows nonlinear boundary
    ↓
ν controls outlier/boundary behavior
    ↓
New Point
    ↓
Decision Score
    ↓
+1 → Normal
-1 → Anomaly
```

### Hyperparameters

| Parameter | Controls                              |
| --------- | ------------------------------------- |
| `kernel`  | Type/shape of boundary                |
| `nu`      | Allowed outliers + boundary behavior  |
| `gamma`   | Local influence / boundary complexity |
